In [1]:
import numpy as np
import math
from qiskit.primitives import Sampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.applications import Tsp

# --- Utility Functions ---
def get_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def sweep_decomposition(nodes, Nv, C):
    depot = nodes[0]
    customers = nodes[1:]
    customer_data = []
    for i, coords in enumerate(customers):
        angle = math.atan2(coords[1], coords[0])
        customer_data.append({'coords': coords, 'angle': angle})
    
    # Sort by polar angle
    customer_data.sort(key=lambda x: x['angle'])
    
    clusters = []
    for i in range(0, len(customer_data), C):
        if len(clusters) < Nv:
            chunk = customer_data[i : i + C]
            clusters.append([depot] + [c['coords'] for c in chunk])
    return clusters

# --- Instance Library ---
instances = {
    1: {"Nv": 2, "C": 5, "nodes": [(0,0), (-2,2), (-5,8), (2,3)]},
    2: {"Nv": 2, "C": 2, "nodes": [(0,0), (-2,2), (-5,8), (2,3)]},
    3: {"Nv": 3, "C": 2, "nodes": [(0,0), (-2,2), (-5,8), (2,3), (5,7), (2,4), (2,-3)]},
    4: {"Nv": 4, "C": 3, "nodes": [(0,0), (-2,2), (-5,8), (6,3), (4,4), (3,2), (0,2), (-2,3), (-4,3), (2,3), (2,7), (-2,5), (-1,4)]}
}

ImportError: cannot import name 'Sampler' from 'qiskit.primitives' (/home/jovyan/QuantumCT-RTRC-qBraid-Challenge/.venv/lib/python3.11/site-packages/qiskit/primitives/__init__.py)

In [ ]:
# --- SET YOUR INSTANCE HERE ---
selected_id = 4 

# Load the data
config = instances[selected_id]
Nv = config["Nv"]
C = config["C"]
nodes = config["nodes"]

print(f"Loaded Instance {selected_id}: {len(nodes)-1} customers, {Nv} vehicles, Capacity {C}")

In [ ]:
# Initialize Quantum Components
sampler = Sampler()
optimizer = COBYLA()
qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=2)
quantum_solver = MinimumEigenOptimizer(qaoa)

# 1. Classical Decomposition (The Sweep)
clusters = sweep_decomposition(nodes, Nv, C)
total_distance = 0

print(f"Starting Hybrid Solve for {len(clusters)} clusters...")

# 2. Quantum Routing Loop
for idx, cluster_nodes in enumerate(clusters):
    n_tsp = len(cluster_nodes)
    
    # Build the local distance matrix for this cluster
    adj_matrix = np.zeros((n_tsp, n_tsp))
    for i in range(n_tsp):
        for j in range(n_tsp):
            adj_matrix[i,j] = get_distance(cluster_nodes[i], cluster_nodes[j])
    
    # Map to TSP Application
    tsp_app = Tsp(adj_matrix)
    qp = tsp_app.to_quadratic_program()
    
    # Solve via QAOA
    result = quantum_solver.solve(qp)
    route = tsp_app.interpret(result)
    
    # Output Results
    total_distance += result.fval
    print(f"  Route {idx+1}: {route} | Distance: {result.fval:.2f}")

print("-" * 30)
print(f"FINAL TOTAL DISTANCE: {total_distance:.2f}")